# Section 4 — Neville's Method

[← Divided Differences](03_newton_divided_differences.ipynb) ·
[Module 2](README.md) ·
[Next: Least Squares →](05_least_squares_best_fit.ipynb)

Neville's Method evaluates the interpolating polynomial at **one requested
point** without first writing down the entire polynomial.

That makes it different from Lagrange or Newton interpolation:

- Lagrange gives an explicit formula for the polynomial;
- Newton gives an explicit polynomial in divided-difference form;
- Neville recursively builds the value $P(x)$ at one chosen $x$.

Suppose $P_{i,j}(x)$ denotes the polynomial that interpolates the data at

$$
x_i,x_{i+1},\ldots,x_j.
$$

The starting values are

$$
P_{i,i}(x)=f(x_i).
$$

For $i<j$,

$$
P_{i,j}(x)
=
\frac{
(x-x_j)P_{i,j-1}(x)
-
(x-x_i)P_{i+1,j}(x)
}{
x_i-x_j
}.
$$

The final answer is $P_{0,n}(x)$.

In [1]:
import math

import numpy as np

## Example from the course review

Let

$$
f(x)=2^{x/2}
$$

and use

$$
x_0=0,\quad x_1=1,\quad x_2=2,\quad x_3=3.
$$

Use Neville's Method to approximate $f(1.5)$.

The first column of the table is simply the function values:

$$
P_{0,0}=f(x_0),\quad
P_{1,1}=f(x_1),\quad
P_{2,2}=f(x_2),\quad
P_{3,3}=f(x_3).
$$

The next column combines pairs of one-point interpolants. Then the next column
combines those two-point interpolants. The upper-right entry is the value of the
full cubic interpolant at $x=1.5$.

In [2]:
def f(x):
    return 2.0 ** (x / 2.0)


def neville_table(x_nodes, y_nodes, x_value):
    """Build Neville's interpolation table at one x-value."""
    n = len(x_nodes)
    table = np.full((n, n), np.nan)
    table[:, 0] = y_nodes

    for column in range(1, n):
        for row in range(n - column):
            x_i = x_nodes[row]
            x_j = x_nodes[row + column]

            left = (x_value - x_j) * table[row, column - 1]
            right = (x_value - x_i) * table[row + 1, column - 1]

            table[row, column] = (left - right) / (x_i - x_j)

    return table


x_nodes = np.array([0.0, 1.0, 2.0, 3.0])
y_nodes = f(x_nodes)
x_value = 1.5

table = neville_table(x_nodes, y_nodes, x_value)
print(table)

[[1.         1.62132034 1.68566017 1.68121843]
 [1.41421356 1.70710678 1.6767767         nan]
 [2.         1.58578644        nan        nan]
 [2.82842712        nan        nan        nan]]


### Reading the table

Column 0 contains $P_{i,i}(1.5)$.

Column 1 contains

$$
P_{0,1}(1.5),\quad
P_{1,2}(1.5),\quad
P_{2,3}(1.5).
$$

Column 2 contains

$$
P_{0,2}(1.5),\quad
P_{1,3}(1.5).
$$

The final entry

$$
P_{0,3}(1.5)
$$

uses all four nodes.

In [3]:
for row in range(len(x_nodes)):
    entries = []

    for column in range(len(x_nodes) - row):
        value = table[row, column]
        entries.append(f"{value:12.8f}")

    print("  ".join(entries))

approximation = table[0, -1]
exact_value = f(x_value)
actual_error = abs(exact_value - approximation)

print()
print(f"Neville approximation = {approximation:.10f}")
print(f"exact value           = {exact_value:.10f}")
print(f"actual error          = {actual_error:.3e}")

  1.00000000    1.62132034    1.68566017    1.68121843
  1.41421356    1.70710678    1.67677670
  2.00000000    1.58578644
  2.82842712

Neville approximation = 1.6812184335
exact value           = 1.6817928305
actual error          = 5.744e-04


## Error bound

Neville's Method evaluates the **same interpolating polynomial** as Lagrange or
Newton interpolation, so it uses the same interpolation remainder:

$$
f(x)-P_3(x)
=
\frac{f^{(4)}(\xi)}{4!}
(x-x_0)(x-x_1)(x-x_2)(x-x_3).
$$

For

$$
f(x)=2^{x/2},
$$

$$
f^{(4)}(x)
=
\left(\frac{\ln 2}{2}\right)^4 2^{x/2}.
$$

On $[0,3]$ this derivative is increasing, so its maximum occurs at $x=3$.

In [4]:
m_value = (math.log(2.0) / 2.0) ** 4 * f(3.0)
product = np.prod(x_value - x_nodes)
error_bound = m_value * abs(product) / math.factorial(4)

print(f"error bound  = {error_bound:.3e}")
print(f"actual error = {actual_error:.3e}")

error bound  = 9.564e-04
actual error = 5.744e-04


### Important interpretation

Neville's Method is **not a different interpolating polynomial**. It is a
different way to evaluate the unique degree-at-most-$n$ interpolating polynomial.
For a single evaluation point, it avoids constructing symbolic basis
polynomials.

### References

- Eric Sullivan, *Numerical Methods: An Inquiry-Based Approach with Python*.
- Leon Q. Brin, *Tea Time Numerical Analysis*.
- Richard L. Burden and J. Douglas Faires, *Numerical Analysis*.
- MAD 4401 course lecture notes and review materials.


[Next: Least Squares →](05_least_squares_best_fit.ipynb)